## Exits evacuation

In [ ]:
# Set parent as root and import config
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))
import configs.simulation_config as cfg
from configs.paths import PROCESSED_DIR, RAW_DIR

import osmnx as ox
import pandas as pd
import geopandas as gpd

In [ ]:
def load_exits(csv_path):
    """
    Load exits from CSV and return GeoDataFrame (EPSG:4326).
    Expected columns: name, lat, lon
    """
    exits = pd.read_csv(csv_path)

    required_cols = {"name", "lat", "lon"}
    missing = required_cols - set(exits.columns)
    if missing:
        raise ValueError(f"Exit file missing columns: {missing}")

    exits["lat"] = pd.to_numeric(exits["lat"], errors="coerce")
    exits["lon"] = pd.to_numeric(exits["lon"], errors="coerce")
    exits = exits.dropna(subset=["lat", "lon"])

    exits_gdf = gpd.GeoDataFrame(
        exits,
        geometry=gpd.points_from_xy(exits["lon"], exits["lat"]),
        crs="EPSG:4326",
    )

    return exits_gdf

In [ ]:
def attach_exits_to_nodes(graph_proj, exits_gdf):
    """
    Snap exits to nearest OSMnx nodes using OSMnx native method.
    """
    exits_proj = exits_gdf.to_crs(graph_proj.graph["crs"])
    xs = exits_proj.geometry.x
    ys = exits_proj.geometry.y

    nearest_nodes = ox.distance.nearest_nodes(graph_proj, X=xs, Y=ys)

    exit_map = {}
    for node_id, name in zip(
        nearest_nodes, 
        exits_gdf["name"], 
    ):
        exit_map[node_id] = name

    nodes, edges = ox.graph_to_gdfs(graph_proj)

    nodes["is_exit"] = False
    nodes["exit_names"] = ""

    for node_id, name in exit_map.items():
        if node_id in nodes.index:
            nodes.loc[node_id, "is_exit"] = True
            nodes.loc[node_id, "exit_names"] = name

    return nodes, edges

In [ ]:
graph = ox.load_graphml(PROCESSED_DIR / "hatyai_graph_with_pop.graphml")

if cfg.ENABLE_EXIT_EVACUATION:
    exits_gdf = load_exits(RAW_DIR / cfg.EXITS_CSV_PATH)
    print(exits_gdf.head())
    exits_gdf.to_file(PROCESSED_DIR / "hatyai_exits.geojson", driver="GeoJSON")
    
    graph_proj = ox.project_graph(graph)
    nodes, edges = attach_exits_to_nodes(graph_proj, exits_gdf)
    print(f"[Exits] Attached to {nodes['is_exit'].sum()} nodes")
    
    graph = ox.graph_from_gdfs(nodes, edges)
    graph = ox.project_graph(graph, to_crs="EPSG:4326")
    ox.save_graphml(graph, PROCESSED_DIR / "hatyai_graph_with_exits.graphml")
else:
    print("[Exits] — skipping exit attachment.")

ox.save_graphml(graph, PROCESSED_DIR / "hatyai_graph_with_dest.graphml")